# HStream Extractor (Colab)

Bulk downloader for [hstream.moe](https://hstream.moe). Each series is saved in its own folder.

**Credits:** [hanime-plugin](https://github.com/cynthia2006/hanime-plugin)


In [ ]:
# @title 1 · Install dependencies
import os, subprocess, requests, glob, re
from urllib.parse import unquote
from tqdm.notebook import tqdm
print('Installing...')
subprocess.run(['pip', 'install', '-q', '--upgrade', 'yt-dlp', 'requests', 'tqdm', 'hanime-plugin'], check=False)
subprocess.run(['apt-get', 'update', '-qq'], check=False)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'aria2', 'ffmpeg'], check=False)
if subprocess.run(['which', 'deno'], capture_output=True).returncode != 0:
    subprocess.run('curl -fsSL https://deno.land/install.sh | sh', shell=True, check=False)
    os.environ['PATH'] = os.path.expanduser('~/.deno/bin') + os.pathsep + os.environ.get('PATH', '')
print('OK')


In [ ]:
# @title 2 · Settings
URL_LIST = "https://hstream.moe/hentai/modaete-yo-adam-kun-1 https://hstream.moe/hentai/modaete-yo-adam-kun-2"  #@param {type:"string"}
DESTINATION_FOLDER = "/content/downloads"  #@param {type:"string"}
XSRF_TOKEN = ""  #@param {type:"string"}
HSTREAM_SESSION = ""  #@param {type:"string"}
SERIES_SLUG = ""  #@param {type:"string"}
YEAR = "2024"  #@param {type:"string"}
print('Settings loaded')


In [ ]:
# @title 3 · Download + mux subtitles
os.makedirs(DESTINATION_FOLDER, exist_ok=True)
deno_bin = os.path.expanduser("~/.deno/bin")
if os.path.isdir(deno_bin):
    os.environ["PATH"] = deno_bin + os.pathsep + os.environ.get("PATH", "")
urls = [u.strip() for u in URL_LIST.replace("\n", " ").split() if u.strip()]
print(f"Found {len(urls)} links\n")
cookie_parts = []
if XSRF_TOKEN.strip(): cookie_parts.append(f"XSRF-TOKEN={XSRF_TOKEN.strip()}")
if HSTREAM_SESSION.strip(): cookie_parts.append(f"hstream_session={HSTREAM_SESSION.strip()}")
COOKIE_HEADER = "; ".join(cookie_parts)
SUB_HOSTS = ["https://oppai-str.shoujo-h.org", "https://imoto-str.ane-h.xyz", "https://shinobu-str.rorikon-h.xyz"]
yp = YEAR.strip() or "2024"
YEARS = []
for y in (yp, "2026", "2025", "2024", "2023", "2022", "2021"):
    if y not in YEARS: YEARS.append(y)
FORMAT_TRIES = ["best", "bestvideo*+bestaudio/best", "best[height<=2160]", "best[height<=1080]", "best[height<=720]"]
def series_folder_name(url):
    token = url.rstrip("/").split("/")[-1]
    name = re.sub(r"-\d+$", "", token)
    return re.sub(r'[\\/:*?"<>|]+', "", name).strip() or "unknown"
def resolve_subtitle_url(page_url):
    headers = {"User-Agent": "Mozilla/5.0", "Referer": "https://hstream.moe/"}
    if COOKIE_HEADER: headers["Cookie"] = COOKIE_HEADER
    html = ""
    try:
        r = requests.get(page_url, headers=headers, timeout=30)
        if r.status_code == 200:
            html = r.text
            found = []
            for pat in [r'href=["\'](https?://[^"\']+?/eng\.ass)["\']', r'href=["\'](https?://[^"\']+?\.ass)["\']']:
                for m in re.finditer(pat, html, re.I):
                    if m.group(1) not in found: found.append(m.group(1))
            for u in found:
                if "eng.ass" in u.lower():
                    tqdm.write(f"  page subtitle: {u}"); return u
            if found:
                tqdm.write(f"  page subtitle: {found[0]}"); return found[0]
            tqdm.write("  no .ass on page")
        else: tqdm.write(f"  page HTTP {r.status_code}")
    except Exception as e: tqdm.write(f"  page scrape failed: {e}")
    try:
        m = re.search(r'id=["\']e_id["\'][^>]*value=["\']([^"\']+)["\']|value=["\']([^"\']+)["\'][^>]*id=["\']e_id["\']', html or "", re.I)
        e_id = (m.group(1) or m.group(2)) if m else None
        if not e_id:
            tqdm.write("  no e_id"); return None
        api = dict(headers)
        api["Content-Type"] = "application/json"
        api["X-Requested-With"] = "XMLHttpRequest"
        if COOKIE_HEADER:
            for part in COOKIE_HEADER.split(";"):
                part = part.strip()
                if part.upper().startswith("XSRF-TOKEN="):
                    api["X-XSRF-TOKEN"] = unquote(part.split("=", 1)[1]); break
        resp = requests.post("https://hstream.moe/player/api", headers=api, json={"episode_id": e_id}, timeout=30)
        if resp.status_code != 200:
            resp = requests.post("https://hstream.moe/player/api", headers=api, data={"episode_id": e_id}, timeout=30)
        if resp.status_code != 200:
            tqdm.write(f"  player API HTTP {resp.status_code}"); return None
        data = resp.json()
        stream_url = data.get("stream_url") or ""
        domains = data.get("stream_domains") or []
        if isinstance(domains, str): domains = [domains]
        if not stream_url or not domains:
            tqdm.write("  player API missing fields"); return None
        domain = domains[0]
        if not str(domain).startswith("http"): domain = "https://" + str(domain).lstrip("/")
        sub = f"{str(domain).rstrip('/')}/{stream_url.strip('/')}/eng.ass"
        tqdm.write(f"  player API subtitle: {sub}"); return sub
    except Exception as e:
        tqdm.write(f"  player API failed: {e}"); return None
def try_download_sub(sub_url, sub_path):
    try:
        r = requests.get(sub_url, stream=True, timeout=30)
        if r.status_code != 200: return False
        total = int(r.headers.get("content-length", 0))
        with open(sub_path, "wb") as f, tqdm(desc="Subtitle", total=total, unit="B", unit_scale=True, unit_divisor=1024, leave=False) as bar:
            for chunk in r.iter_content(1024):
                bar.update(len(chunk)); f.write(chunk)
        return True
    except Exception: return False
def run_ytdlp(url, out, fmt, dl):
    cmd = ["yt-dlp", "-f", fmt, "--downloader", dl, "--concurrent-fragments", "8", "-o", out, "--no-mtime", "--retries", "5", "--fragment-retries", "5"]
    if dl == "aria2c": cmd += ["--downloader-args", "aria2c:-x 16 -s 16 -k 1M"]
    if COOKIE_HEADER: cmd += ["--add-header", f"Cookie: {COOKIE_HEADER}"]
    cmd.append(url)
    return subprocess.run(cmd, check=True, capture_output=True, text=True)
for i, url in enumerate(tqdm(urls, desc="Overall", unit="video"), 1):
    tqdm.write(f"\n[{i}/{len(urls)}] {url}")
    series_dir = os.path.join(DESTINATION_FOLDER, series_folder_name(url))
    os.makedirs(series_dir, exist_ok=True)
    tqdm.write(f"Series folder: {series_dir}")
    out = os.path.join(series_dir, "%(title)s.%(ext)s")
    ok = False
    for fmt in FORMAT_TRIES:
        for dl in ("aria2c", "ffmpeg"):
            try:
                tqdm.write(f"  try {fmt} / {dl}")
                run_ytdlp(url, out, fmt, dl); ok = True; break
            except subprocess.CalledProcessError as e:
                err = e.stderr or e.stdout or ""
                for line in err.strip().splitlines()[-6:]:
                    if "ERROR" in line or "404" in line: tqdm.write(line)
        if ok: break
    if not ok:
        tqdm.write("download failed"); continue
    files = [f for f in glob.glob(os.path.join(series_dir, "*")) if not f.endswith(".ass") and "-sample" not in f.lower() and os.path.isfile(f)]
    if not files: continue
    latest = max(files, key=os.path.getctime)
    base = os.path.splitext(os.path.basename(latest))[0]
    final_mkv = os.path.join(series_dir, f"{base}.mkv")
    if latest.endswith(".mkv"):
        tqdm.write(f"Already MKV: {latest}"); continue
    ep = url.rstrip("/").split("/")[-1].split("-")[-1]
    slug = "-".join(url.rstrip("/").split("/")[-1].split("-")[:-1])
    sub_path = os.path.join(series_dir, f"{base}.ass")
    sub_ok = False
    tqdm.write("Resolving subtitle...")
    live = resolve_subtitle_url(url)
    if live and try_download_sub(live, sub_path): sub_ok = True
    if not sub_ok:
        particles = {"no","wa","wo","ga","ni","de","to","na","o","yo","kun","chan","san"}
        parts = slug.split("-")
        cands = []
        if SERIES_SLUG.strip(): cands.append(SERIES_SLUG.strip())
        cands.append(".".join(parts))
        cands.append(".".join(w if w in particles else w.capitalize() for w in parts))
        glued, j = [], 0
        while j < len(parts):
            w = parts[j]
            if j+1 < len(parts) and parts[j+1] in {"kun","chan","san"} and w not in particles:
                glued.append(w.capitalize() + parts[j+1]); j += 2
            else:
                glued.append(w if w in particles else w.capitalize()); j += 1
        cands.append(".".join(glued)); cands.append(slug)
        cands = list(dict.fromkeys(cands))
        for host in SUB_HOSTS:
            for y in YEARS:
                for s in cands:
                    su = f"{host}/{y}/{s}/E{int(ep):02d}/eng.ass"
                    if try_download_sub(su, sub_path):
                        sub_ok = True; tqdm.write(f"  found: {su}"); break
                if sub_ok: break
            if sub_ok: break
    if sub_ok:
        try:
            subprocess.run(["ffmpeg","-y","-i",latest,"-i",sub_path,"-map","0","-map","1","-c","copy","-metadata:s:s:0","language=eng",final_mkv], check=True)
            if os.path.exists(sub_path): os.remove(sub_path)
            if latest != final_mkv and os.path.exists(latest): os.remove(latest)
            tqdm.write(f"Saved: {final_mkv}")
        except Exception as ex: tqdm.write(f"remux error: {ex}")
    else: tqdm.write("No subtitle; kept original")
print("\nDONE")


In [ ]:
# @title 4 · Make sample from one file
FILE_PATH = ""  #@param {type:"string"}
SAMPLE_START = "00:12:01"  #@param {type:"string"}
SAMPLE_DURATION_SEC = 60  #@param {type:"integer"}
from pathlib import Path
def parse_ts(ts):
    p = [int(x) for x in ts.strip().split(':')]
    if len(p) == 3: return p[0]*3600 + p[1]*60 + p[2]
    if len(p) == 2: return p[0]*60 + p[1]
    return p[0]
def fmt_ts(t):
    h, r = divmod(max(0, t), 3600)
    m, s = divmod(r, 60)
    return f'{h:02d}:{m:02d}:{s:02d}' if h else f'{m:02d}:{s:02d}'
path = Path(FILE_PATH.strip())
if not FILE_PATH.strip():
    print('Set FILE_PATH to a video under /content/downloads/...')
elif not path.is_file():
    print(f'File not found: {path}')
else:
    st = parse_ts(SAMPLE_START); dur = int(SAMPLE_DURATION_SEC)
    sl, el = fmt_ts(st), fmt_ts(st + dur)
    mins = max(1, round(dur / 60))
    out = path.parent / f'{path.stem}-sample [{sl} - {el}] {mins} Minute{path.suffix}'
    print(f'{path.name} -> {out.name}')
    try:
        subprocess.run(['ffmpeg', '-y', '-ss', str(st), '-i', str(path), '-t', str(dur), '-c', 'copy', str(out)], check=True, capture_output=True)
        print(f'OK: {out}')
    except subprocess.CalledProcessError:
        print('stream copy failed, re-encoding...')
        try:
            subprocess.run(['ffmpeg', '-y', '-ss', str(st), '-i', str(path), '-t', str(dur), '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '23', '-c:a', 'aac', '-b:a', '128k', str(out)], check=True, capture_output=True)
            print(f'OK (re-encode): {out}')
        except Exception as e:
            print(f'FAILED: {e}')


In [ ]:
# @title 5 · Zip downloads
!zip -r /content/hstream_downloads.zip {DESTINATION_FOLDER}
print('Created: /content/hstream_downloads.zip')
print('Download from left sidebar → Files')
